# FMA metadata exploration

Open this notebook with the **Music Atlas** kernel and run all cells. It reads the shared FMA metadata and inventory and shows the results inline.

The default is the pre-audit inventory. After the audit, set `USE_AUDIT = True` to keep valid track IDs from `manifests/tracks.parquet`. Durations come from the published metadata in both runs.

In [ ]:
from ast import literal_eval

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from music_atlas.settings import paths

USE_AUDIT = False

data = paths()["data"]
metadata = data / "fma/fma_metadata"
inventory = pd.read_csv(
    data / "manifests/fma_full_inventory.csv", usecols=["track_id", "status"], index_col="track_id"
)
assert inventory.index.is_unique
ids = inventory.index[inventory["status"].eq("present")]
if USE_AUDIT:
    audit = pd.read_parquet(data / "manifests/tracks.parquet", columns=["track_id", "valid"])
    assert audit["track_id"].is_unique and pd.api.types.is_bool_dtype(audit["valid"])
    assert audit["valid"].notna().all()
    ids = ids.intersection(audit.loc[audit["valid"], "track_id"])

assert len(ids), "The selected catalogue is empty."
fields = {
    ("artist", "id"): "artist_id",
    ("album", "id"): "album_id",
    ("set", "split"): "split",
    ("track", "duration"): "duration",
    ("track", "genres"): "genres",
    ("track", "genres_all"): "genres_all",
}
tracks = pd.read_csv(metadata / "tracks.csv", index_col=0, header=[0, 1], low_memory=False)
assert tracks.index.is_unique
missing_rows = len(ids.difference(tracks.index))
tracks = tracks[list(fields)].reindex(ids)
tracks.columns = list(fields.values())
for field in ["artist_id", "album_id"]:
    tracks[field] = tracks[field].where(tracks[field] > 0)

catalogue = "audited" if USE_AUDIT else "pre-audit"
print(f"{len(tracks):,} tracks ({catalogue}), {missing_rows:,} without a metadata row.")

## Genres and their hierarchy

FMA provides direct tags in `genres` and tags with their ancestors in `genres_all`. A track counts once per genre. The full list shows each genre's parent and top-level branch. [FMA metadata preparation](https://github.com/mdeff/fma/blob/master/creation.ipynb)

In [ ]:
genres = pd.read_csv(metadata / "genres.csv", index_col="genre_id")
labels = tracks[["genres", "genres_all"]].fillna("[]").map(literal_eval)
direct = labels["genres"].explode().dropna().reset_index(name="genre_id").drop_duplicates()
inclusive = labels["genres_all"].explode().dropna().reset_index(name="genre_id").drop_duplicates()
assert (
    direct["genre_id"].isin(genres.index).all() and inclusive["genre_id"].isin(genres.index).all()
)

genres["Direct"] = direct["genre_id"].value_counts().reindex(genres.index, fill_value=0)
genres["With descendants"] = (
    inclusive["genre_id"].value_counts().reindex(genres.index, fill_value=0)
)
assert genres["With descendants"].ge(genres["Direct"]).all()

listing = genres.assign(
    Parent=genres["parent"].map(genres["title"]).fillna(""),
    Branch=genres["top_level"].map(genres["title"]),
).sort_values(["Branch", "parent", "title"])
listing = listing[["title", "Parent", "Branch", "Direct", "With descendants"]]
display(
    HTML(
        f"<details><summary>Full FMA genre list ({len(listing)})</summary>"
        f"{listing.rename(columns={'title': 'Genre'}).to_html(index=True)}</details>"
    )
)

roots = genres[genres["parent"].eq(0)]
children = genres[genres["parent"].ne(0)].nlargest(10, "With descendants")
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
for ax, rows, title in zip(
    axes, [roots, children], ["Top-level genres", "Most frequent subgenres"], strict=True
):
    rows.sort_values("With descendants").set_index("title")[
        ["Direct", "With descendants"]
    ].plot.barh(ax=ax, title=title, xlabel="Unique tracks", ylabel="", legend=ax is axes[0])
plt.show()

In [ ]:
common = genres[genres["Direct"].gt(0)].nlargest(10, "Direct").index
if len(common) >= 2:
    selected = direct[direct["genre_id"].isin(common)]
    flags = pd.crosstab(selected["track_id"], selected["genre_id"]).reindex(columns=common)
    flags = flags.to_numpy(dtype=np.int64)
    shared = flags.T @ flags
    totals = shared.diagonal()
    similarity = shared / (totals + totals[:, None] - shared)
    np.fill_diagonal(similarity, np.nan)

    fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
    cmap = plt.get_cmap("Blues").with_extremes(bad="#e5e7eb")
    heatmap = ax.imshow(similarity, cmap=cmap, vmin=0, vmax=np.nanmax(similarity) or 1)
    ax.set_xticks(range(len(common)), genres.loc[common, "title"], rotation=45, ha="right")
    ax.set_yticks(range(len(common)), genres.loc[common, "title"])
    ax.set_title("Co-occurrence of direct genre labels")
    fig.colorbar(heatmap, ax=ax, label="Shared / union of tracks")
    plt.show()
else:
    print("Co-occurrence requires at least two observed genres.")

## Concentration, duration and coverage

In [ ]:
root_tags = inclusive[inclusive["genre_id"].isin(roots.index)]
roots_per_track = root_tags.groupby("track_id").size().reindex(ids, fill_value=0)
labels_per_track = direct.groupby("track_id").size().reindex(ids, fill_value=0)
durations = tracks["duration"].dropna()
durations = durations[durations.gt(0) & np.isfinite(durations)]

fig, (ax, duration_ax) = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
for field, name in [("artist_id", "Artists"), ("album_id", "Albums")]:
    shares = tracks[field].value_counts().cumsum().reset_index(drop=True) / len(tracks)
    ax.plot(np.arange(1, len(shares) + 1), shares * 100, label=name)
ax.set(
    xscale="log",
    xlabel="Rank by track count",
    ylabel="Cumulative catalogue share (%)",
    title="Artist and album concentration",
)
ax.legend()
if not durations.empty:
    duration_ax.hist(
        durations, bins=np.geomspace(durations.min() * 0.99, durations.max() * 1.01, 36)
    )
    duration_ax.axvline(
        durations.median(), color="tab:orange", label=f"Median {durations.median():g} s"
    )
    duration_ax.legend()
duration_ax.set(
    xscale="log", xlabel="Published duration (seconds)", ylabel="Tracks", title="Duration"
)
plt.show()

coverage = pd.Series(
    {
        "No genre tags": labels_per_track.eq(0).sum(),
        "One direct tag": labels_per_track.eq(1).sum(),
        "Multiple direct tags": labels_per_track.gt(1).sum(),
        "Multiple top-level genres": roots_per_track.gt(1).sum(),
        "Missing artist ID": tracks["artist_id"].isna().sum(),
        "Missing album ID": tracks["album_id"].isna().sum(),
        "Missing or invalid duration": len(tracks) - len(durations),
    }
)
ax = (coverage / len(tracks) * 100).plot.barh(
    figsize=(8, 3), xlim=(0, 120), title="Annotation coverage"
)
ax.set_xlabel("Share of selected tracks (%)")
ax.set_xticks(range(0, 101, 20))
ax.invert_yaxis()
ax.bar_label(ax.containers[0], labels=[f"{n:,}" for n in coverage], padding=4)
plt.tight_layout()
plt.show()

tagged = direct.join(tracks["artist_id"], on="track_id")
artist_counts = tagged.groupby(["genre_id", "artist_id"]).size().groupby(level=0)
support = (
    genres[["title", "Direct"]]
    .query("Direct > 0")
    .assign(
        Artists=artist_counts.size().reindex(genres.index, fill_value=0),
        Top_artist_pct=(100 * artist_counts.max() / genres["Direct"]).round(1),
    )
)
display(support.nsmallest(5, ["Artists", "Direct"]))
display(support.query("Direct >= 20").nlargest(3, "Top_artist_pct"))

for field in ["artist_id", "album_id"]:
    overlap = tracks.groupby(field)["split"].nunique().gt(1).sum()
    print(f"{field}: {overlap:,} IDs span multiple published splits.")

## A candidate with nine genre groups

**Soul-RnB** is FMA's soul and rhythm-and-blues branch. Its descendants include Funk, Deep Funk and Disco. This candidate groups it with Jazz and Blues, whose musical traditions helped shape R&B. [Library of Congress](https://www.loc.gov/collections/songs-of-america/articles-and-essays/musical-styles/popular-songs-of-the-day/rhythm-and-blues/)

**Easy Listening** historically includes light arrangements for orchestras and small ensembles. FMA's branch includes Lounge, Nu-Jazz and vocal easy listening. We keep its name in a shared group with Instrumental. [Museum of Broadcast Communications](https://www.museum.tv/portfolio-2/easy-listening/beautiful-music-format)

The mapping assigns every FMA top-level branch to one group. A track can still belong to several groups. Use the track and artist counts to evaluate the proposal. The Folk, Country & Regional group combines several distinct traditions and needs a closer look before adoption.

In [ ]:
groups = {
    "Rock & Pop": (12, 10),
    "Electronic": (15,),
    "Hip-Hop": (21,),
    "Jazz, Blues & Soul/R&B": (4, 3, 14),
    "Folk, Country & Regional": (17, 9, 2, 8),
    "Classical": (5,),
    "Instrumental & Easy Listening": (1235, 13),
    "Experimental": (38,),
    "Spoken": (20,),
}
group_for_root = {root: group for group, root_ids in groups.items() for root in root_ids}
assert set(group_for_root) == set(roots.index)
assert len(group_for_root) == sum(map(len, groups.values()))

group_tags = root_tags.assign(Group=root_tags["genre_id"].map(group_for_root))
group_tags = group_tags.drop_duplicates(["track_id", "Group"]).join(
    tracks["artist_id"], on="track_id"
)
group_summary = group_tags.groupby("Group").agg(
    Tracks=("track_id", "size"), Artists=("artist_id", "nunique")
)
group_summary = group_summary.reindex(groups, fill_value=0)
group_summary["FMA branches"] = [
    " + ".join(genres.loc[list(group), "title"]) for group in groups.values()
]
display(group_summary[["FMA branches", "Tracks", "Artists"]])

groups_per_track = group_tags.groupby("track_id").size().reindex(ids, fill_value=0)
assert groups_per_track.le(roots_per_track).all()
assert groups_per_track.gt(0).equals(roots_per_track.gt(0))
print(
    f"Tracks with multiple labels: {roots_per_track.gt(1).sum():,} original, "
    f"{groups_per_track.gt(1).sum():,} with the proposed groups."
)

## Interpretation

In the pre-audit catalogue, Rock has 8,038 direct tags and 32,923 tracks including subgenres. Deep Funk has one track from one artist. Decide whether to score rare labels only after checking their support in independent artist and duplicate groups.

There are 2,231 untagged tracks, 1,025 missing album IDs and 16 missing or invalid published durations. Keep untagged tracks out of supervised evaluation. Use multilabel targets for the genre groups and reassess the mapping after the audit.

The published split shares 670 albums across partitions. Keep each artist and each confirmed duplicate group in one partition, check album overlap and report per-class support before fixing the split.